In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/Datasets_for_fyp"
csv_files = [f for f in os.listdir(DATASET_DIR) if f.endswith(".csv")]

print("CSV files found:", csv_files)


CSV files found: ['dataset2.csv', 'dataset1.csv', 'dataset3.csv', 'dataset4.csv', 'Phishing_Legitimate_full.csv', 'dataset_phishing.csv', 'Website Phishing.csv', 'Zieni_Dataset for phishing detection.csv', 'Dataset.csv', 'StealthPhisher2025.csv', 'new_data_urls.csv']


In [ ]:
import pandas as pd

for file in csv_files:
    df_temp = pd.read_csv(os.path.join(DATASET_DIR, file))
    print("\n==============================")
    print(f"File: {file}")
    print("Columns:", df_temp.columns.tolist())
    print("Shape:", df_temp.shape)



File: dataset2.csv
Columns: ['qty_dot_url', 'qty_hyphen_url', 'qty_underline_url', 'qty_slash_url', 'qty_questionmark_url', 'qty_equal_url', 'qty_at_url', 'qty_and_url', 'qty_exclamation_url', 'qty_space_url', 'qty_tilde_url', 'qty_comma_url', 'qty_plus_url', 'qty_asterisk_url', 'qty_hashtag_url', 'qty_dollar_url', 'qty_percent_url', 'qty_tld_url', 'length_url', 'qty_dot_domain', 'qty_hyphen_domain', 'qty_underline_domain', 'qty_slash_domain', 'qty_questionmark_domain', 'qty_equal_domain', 'qty_at_domain', 'qty_and_domain', 'qty_exclamation_domain', 'qty_space_domain', 'qty_tilde_domain', 'qty_comma_domain', 'qty_plus_domain', 'qty_asterisk_domain', 'qty_hashtag_domain', 'qty_dollar_domain', 'qty_percent_domain', 'qty_vowels_domain', 'domain_length', 'domain_in_ip', 'server_client_domain', 'qty_dot_directory', 'qty_hyphen_directory', 'qty_underline_directory', 'qty_slash_directory', 'qty_questionmark_directory', 'qty_equal_directory', 'qty_at_directory', 'qty_and_directory', 'qty_excl

load and standardize datasets


In [ ]:
for file in csv_files:
    try:
        df_temp = pd.read_csv(os.path.join(DATASET_DIR, file), header=0, index_col=False)

        # 1. Immediate cleanup
        df_temp = df_temp.loc[:, ~df_temp.columns.duplicated()].copy()
        df_temp.columns = df_temp.columns.str.lower().str.strip()

        # 2. Standardize URL (Handle multiple potential matches)
        url_cols = ["url", "website", "domain", "websites"]
        found_url = [c for c in url_cols if c in df_temp.columns]
        if found_url:
            # Rename the first match to 'url' and drop others if they exist
            df_temp.rename(columns={found_url[0]: "url"}, inplace=True)

        # 3. Standardize Label
        label_cols = ["label", "class", "result", "type", "status"]
        found_label = [c for c in label_cols if c in df_temp.columns]
        if found_label:
            df_temp.rename(columns={found_label[0]: "label"}, inplace=True)

        # 4. Filter and Finalize
        if "url" in df_temp.columns and "label" in df_temp.columns:
            # Keep only the first occurrence of 'url' and 'label' if duplicates exist
            df_temp = df_temp.loc[:, ~df_temp.columns.duplicated()]
            df_temp = df_temp[["url", "label"]].copy()
            df_temp = df_temp.dropna(subset=["url", "label"])

            # Reset index to ensure it's unique before adding to list
            df_temp = df_temp.reset_index(drop=True)
            dfs.append(df_temp)

    except Exception as e:
        print(f"Skipping file {file} due to error: {e}")

In [ ]:
# Create a clean list for finalized dataframes
clean_dfs = []

for temp_df in dfs:
    # 1. Ensure no duplicate column names exist within the individual dataframe
    # This keeps the first occurrence of 'url' or 'label' and drops others
    temp_df = temp_df.loc[:, ~temp_df.columns.duplicated()].copy()

    # 2. Reset the index of the individual dataframe
    temp_df = temp_df.reset_index(drop=True)

    clean_dfs.append(temp_df)

# Combine all datasets
if clean_dfs:
    df = pd.concat(clean_dfs, axis=0, ignore_index=True)
    print("Combined dataset shape:", df.shape)

    # Show unique labels to check for inconsistencies
    print("\nUnique labels found in combined data:")
    print(df['label'].unique())
else:
    print("No valid datasets found to combine.")

Combined dataset shape: (4555104, 2)

Unique labels found in combined data:
['legitimate' 'phishing' 'Phishing' 'Legitimate' 0 1]


In [ ]:
print(df['label'].value_counts())

label
1             1281084
0             1184946
Phishing      1054836
Legitimate     965658
phishing        34290
legitimate      34290
Name: count, dtype: int64


In [ ]:
# Standardizing labels to 0 (benign) and 1 (phishing)
label_mapping = {
    'benign': 0, 'safe': 0, 'legitimate': 0, '0': 0, 0: 0, 'good': 0,
    'phishing': 1, 'malicious': 1, 'bad': 1, '1': 1, 1: 1, 'spam': 1
}

df['label'] = df['label'].map(label_mapping)

# Drop any rows that didn't match our mapping (if any)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

In [ ]:
# Remove duplicate URLs
df = df.drop_duplicates(subset=['url']).reset_index(drop=True)

# Shuffle the data (important for training later)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final dataset size: {df.shape}")

Final dataset size: (808042, 2)


In [ ]:
df.to_csv("/content/drive/MyDrive/Datasets_for_fyp/cleaned_combined_dataset.csv", index=False)

In [ ]:
!pip install tldextract


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 2.1 MB/s eta 0:00:00


In [ ]:
import re
import tldextract


In [ ]:
# Helper functions
def url_length(url): return len(str(url))
def count_dots(url): return str(url).count('.')
def count_hyphens(url): return str(url).count('-')
def count_slashes(url): return str(url).count('/')
def count_at(url): return str(url).count('@')
def count_question(url): return str(url).count('?')
def count_equal(url): return str(url).count('=')
def digit_count(url): return sum(c.isdigit() for c in str(url))
def letter_count(url): return sum(c.isalpha() for c in str(url))
def count_special_chars(url): return len(re.findall(r"[^A-Za-z0-9]", str(url)))
def has_ip_address(url):
    ip_pattern = r"\b(?:\d{1,3}\.){3}\d{1,3}\b"
    return 1 if re.search(ip_pattern, str(url)) else 0
def https_token(url): return 1 if str(url).startswith("https") else 0
def is_encoded(url): return 1 if "%" in str(url) else 0
def extract_domain(url):
    try: return tldextract.extract(str(url)).registered_domain
    except: return ""
def extract_subdomain(url):
    try: return tldextract.extract(str(url)).subdomain
    except: return ""
def domain_length(url): return len(extract_domain(url))
def subdomain_length(url): return len(extract_subdomain(url))
def has_multiple_subdomains(url): return 1 if extract_subdomain(url).count('.') >= 2 else 0
def suspicious_words(url):
    keywords = ["secure", "account", "login", "update", "verify", "bank", "free", "lucky"]
    return sum(1 for k in keywords if k in str(url).lower())
def count_params(url): return str(url).count('&')
def has_shortening_service(url):
    services = ["bit.ly", "tinyurl", "goo.gl", "t.co", "ow.ly"]
    return 1 if any(s in str(url).lower() for s in services) else 0
def get_tld(url):
    try: return tldextract.extract(str(url)).suffix
    except: return ""
def tld_length(url): return len(get_tld(url))


In [ ]:
# Extract features for all URLs
feature_data = {
    "url_length": df["url"].apply(url_length),
    "count_dots": df["url"].apply(count_dots),
    "count_hyphens": df["url"].apply(count_hyphens),
    "count_slash": df["url"].apply(count_slashes),
    "count_at": df["url"].apply(count_at),
    "count_question": df["url"].apply(count_question),
    "count_equal": df["url"].apply(count_equal),
    "digit_count": df["url"].apply(digit_count),
    "letter_count": df["url"].apply(letter_count),
    "special_chars": df["url"].apply(count_special_chars),
    "has_ip": df["url"].apply(has_ip_address),
    "https_flag": df["url"].apply(https_token),
    "encoded_url": df["url"].apply(is_encoded),
    "domain_length": df["url"].apply(domain_length),
    "subdomain_length": df["url"].apply(subdomain_length),
    "multiple_subdomains": df["url"].apply(has_multiple_subdomains),
    "suspicious_words": df["url"].apply(suspicious_words),
    "count_params": df["url"].apply(count_params),
    "shortening_service": df["url"].apply(has_shortening_service),
    "tld_length": df["url"].apply(tld_length)
}

feature_df = pd.DataFrame(feature_data)
print("Feature extraction complete. Shape:", feature_df.shape)


/tmp/ipython-input-415320120.py:18: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  try: return tldextract.extract(str(url)).registered_domain


Feature extraction complete. Shape: (808042, 20)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [ ]:
# Split data
X = feature_df
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [ ]:
#Scale features (optional)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)




In [ ]:
# Train Random Forest
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train_scaled, y_train)



RandomForestClassifier(n_estimators=200, random_state=42)

In [ ]:
# Evaluate
y_pred = clf.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



Accuracy: 0.9379799392360574
              precision    recall  f1-score   support

           0       0.94      0.92      0.93     76203
           1       0.93      0.95      0.94     85406

    accuracy                           0.94    161609
   macro avg       0.94      0.94      0.94    161609
weighted avg       0.94      0.94      0.94    161609



In [ ]:
# Save model, scaler, and features
joblib.dump(clf, "/content/drive/MyDrive/Datasets_for_fyp/rf_model.pkl")
joblib.dump(scaler, "/content/drive/MyDrive/Datasets_for_fyp/scaler.pkl")
joblib.dump(feature_df.columns.tolist(), "/content/drive/MyDrive/Datasets_for_fyp/features.pkl")
print("Model, scaler, and feature list saved.")

Model, scaler, and feature list saved.
